# HiSparse on A3 入门 (1/8)：为什么 A3 需要自己的一套移植

> **一句话结论**：hisparse-a3 不是另一套框架，而是 PR #20926 同一数据面在 Ascend 910_93 上的移植增量；框架契约不变，只把 A5 上由硬件/编译链免费提供的能力逐项换成软件实现。

## 1. 基线与环境

- 基线：`pr20926 @ 5a54eb466`（2026-11-09），仓内只包含在此之上的增量；
- 硬件：Ascend 910_93（Atlas A3），16 die / TP16 / DP1，无 MTP；
- 软件：CANN 9.2.1，Python 3.12.16；
- 模型：GLM-5.2-w4a8c8（同族验证）/ GLM-5.3-Flash（生产）。

## 2. A3 撞上的四个能力缺口

| 缺口 | A5 上的情况 | A3 上的后果 |
| --- | --- | --- |
| SIMT 编译链 | 原生 AscendC SIMT AIV 可编译 | `dav-c220` 无 `asc_simt`，resolver 编译直接失败 |
| fp8 block quant | 硬件原生 E4M3FN | `dst_type=fp8` 报 161002，NoPE 只能改 int8 |
| 原生 int8 KV cache | 硬件可直接落量化行 | 无原生 int8 KV cache，每行都要软件构建 |
| Triton 发射开销 | A5 上可接受 | 2688 program 发射约 25.7–30 ms/步，压关键路径 |

## 3. 一句话理解移植成本

> **A5 的 C8 是硬件免费，A3 的 C8 是软件税。**

A5 上由编译链、硬件算子、发射效率承担的部分，到 A3 上要靠 AICPU 任务、AIV 融合算子、int8 桥接与分块 gather 补回来。

## 4. 不变的部分

三级池职责、656B 行寻址、resolver 算法语义、Core 生命周期、图捕获安全约束，两端完全一致。后面七篇逐个讲四项替换（M1–M4）、前向流程、性能账本与踩坑。

## 5. 小结

- a3 = 同一数据面 + 四项软件替换；
- 先记住四个缺口：SIMT、fp8、原生 int8 KV、Triton 发射；
- 下一篇讲 M1：int8 C8 直读桥。

**下一篇**：`02-m1-int8-c8-bridge.ipynb`
